# DSPy POC: Prompt generation for models on RHOAI

This notebook shows how to use **DSPy** for prompt generation and optimization with a model deployed on **Red Hat OpenShift AI** (RHOAI).

- **Setup:** Configure DSPy to use the RHOAI inference endpoint (OpenAI-compatible API) with bearer token auth.
- **Prompt generation:** Use signatures (e.g. `question -> answer`) and `ChainOfThought`; DSPy builds the prompt and calls your model.
- **Optimization:** Use `BootstrapFewShot` to optimize the prompt (few-shot examples) against a small trainset and metric.

**Prerequisites:** `pip install dspy-ai`. The RHOAI inference endpoint must be **OpenAI-compatible** (e.g. `/v1/chat/completions`). If your deployment uses a different API (e.g. Caikit), you may need an adapter or custom client.

In [10]:
# DSPy POC: Prompt generation for models deployed on RHOAI
# Uses OpenAI-compatible inference API on Red Hat OpenShift AI

import os
from dotenv import load_dotenv
load_dotenv()  # load .env from repo root or current dir (copy from .env.example)

import dspy

# --- RHOAI model configuration ---
# Set in .env (see .env.example) or export RHOAI_INFERENCE_URL, RHOAI_TOKEN
RHOAI_BASE_URL = os.getenv(
    "RHOAI_INFERENCE_URL",
    "",
)
RHOAI_TOKEN = os.getenv("RHOAI_TOKEN", "")  # Set RHOAI_TOKEN or paste token below for POC
if not RHOAI_TOKEN:
    # POC: paste your bearer token here or set RHOAI_TOKEN in the environment
    RHOAI_TOKEN = ""

# OpenAI-compatible endpoint: base URL often needs /v1 for chat completions
api_base = RHOAI_BASE_URL.rstrip("/") + "/v1" if "/v1" not in RHOAI_BASE_URL else RHOAI_BASE_URL.rstrip("/")

lm = dspy.LM(
    "openai/redhataillama-3-1-8b-instruct",
    api_key=RHOAI_TOKEN,
    api_base=api_base,
)
dspy.configure(lm=lm)
print("DSPy configured with RHOAI endpoint:", RHOAI_BASE_URL)

DSPy configured with RHOAI endpoint: https://redhataillama-3-1-8b-instruct-autox.apps.rosa.rhoai-dev.mfe3.p3.openshiftapps.com


## 1. Simple prompt-based prediction (no optimization)

Define a **signature** (input -> output). DSPy generates the prompt and calls the RHOAI model.

In [11]:
# Signature: question -> answer (DSPy will generate the actual prompt text)
class SimpleQA(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought("question -> answer")

    def forward(self, question: str):
        return self.prog(question=question)

qa = SimpleQA()
question = "What is the capital of France?"
result = qa(question=question)
# result is a Prediction with output fields only (reasoning, answer), not the input
print("Question:", question)
print("Reasoning:", result.reasoning)
print("Answer:", result.answer)

Question: What is the capital of France?
Reasoning: The question asks for the capital of France. The capital of France is the city where the country's government is located. After considering various cities in France, such as Paris, Lyon, and Marseille, it is clear that Paris is the seat of the French government.
Answer: Paris


## 2. Prompt optimization with BootstrapFewShot

Use a small **trainset** and a **metric** so DSPy can optimize the prompt (e.g. add few-shot examples) for the model deployed on RHOAI.

In [13]:
# Tiny trainset: (question, answer) for prompt optimization
trainset = [
    dspy.Example(question="What is 2 + 2?", answer="4").with_inputs("question"),
    dspy.Example(question="What is the capital of Italy?", answer="Rome").with_inputs("question"),
    dspy.Example(question="How many continents are there?", answer="7").with_inputs("question"),
]

def metric(example, pred, trace=None):
    """Score 1 if the predicted answer matches the gold answer (case-insensitive)."""
    return 1.0 if pred.answer.strip().lower() == example.answer.strip().lower() else 0.0

# Compile: DSPy optimizes the program (e.g. selects few-shot examples) using the RHOAI model
from dspy.teleprompt import BootstrapFewShot

teleprompter = BootstrapFewShot(metric=metric, max_bootstrapped_demos=2, max_labeled_demos=2)
optimized_qa = teleprompter.compile(qa, trainset=trainset)

# Run optimized program on a new question
test_question = "What is the capital of Germany?"
test_result = optimized_qa(question=test_question)
print("Optimized program - Question:", test_question)
print("Answer:", test_result.answer)

 67%|██████▋   | 2/3 [00:00<00:00, 47.98it/s]

Bootstrapped 2 full traces after 2 examples for up to 1 rounds, amounting to 2 attempts.
Optimized program - Question: What is the capital of Germany?
Answer: Berlin


## 2b. Leaderboard: baseline vs optimized (by eval score)

Evaluate both the **baseline** prompt (no few-shot optimization) and the **optimized** prompt on the same eval set. Rank by score so you can compare effectiveness.

In [14]:
# Eval set: use trainset for POC (or define a separate evalset for production)
evalset = trainset

def run_eval(program, evalset, metric_fn):
    """Run program on evalset and return mean metric score."""
    scores = []
    for ex in evalset:
        try:
            pred = program(question=ex.question)
            s = metric_fn(ex, pred)
            scores.append(float(s) if s is not None else 0.0)
        except Exception as e:
            scores.append(0.0)
    return sum(scores) / len(scores) if scores else 0.0

# Evaluate baseline (no optimization) and optimized program
print("Evaluating baseline (ChainOfThought, no few-shot)...")
score_baseline = run_eval(qa, evalset, metric)
print("Evaluating optimized (BootstrapFewShot)...")
score_optimized = run_eval(optimized_qa, evalset, metric)

# Build leaderboard: rank by score descending
leaderboard = [
    {"config": "Optimized (BootstrapFewShot)", "score": score_optimized, "description": "Few-shot examples + CoT"},
    {"config": "Baseline (no optimization)", "score": score_baseline, "description": "ChainOfThought only"},
]
leaderboard.sort(key=lambda x: x["score"], reverse=True)

# Display ranked table
print("\n" + "=" * 60)
print("LEADERBOARD (ranked by eval score)")
print("=" * 60)
for rank, row in enumerate(leaderboard, 1):
    print(f"  #{rank}  {row['config']:<35}  score = {row['score']:.2%}  ({row['description']})")
print("=" * 60)

Evaluating baseline (ChainOfThought, no few-shot)...
Evaluating optimized (BootstrapFewShot)...

LEADERBOARD (ranked by eval score)
  #1  Optimized (BootstrapFewShot)         score = 100.00%  (Few-shot examples + CoT)
  #2  Baseline (no optimization)           score = 66.67%  (ChainOfThought only)


In [15]:
# Optional: display as a pandas DataFrame (nicer in notebooks)
try:
    import pandas as pd
    df = pd.DataFrame(leaderboard)[["config", "score", "description"]]
    df.insert(0, "rank", range(1, len(df) + 1))
    df["score"] = df["score"].apply(lambda x: f"{x:.2%}")
    display(df)
except ImportError:
    pass

## 3. Inspect the generated prompt

After compilation, you can inspect the prompt (instructions + few-shot examples) that DSPy built for the RHOAI model.

In [16]:
# Show the optimized module's prompt (instructions + demos)
# API varies by DSPy version; use getattr to avoid AttributeError
prog = optimized_qa.prog
sig = getattr(prog, "extended_signature", None) or getattr(prog, "signature", None)
if sig is not None:
    print("Signature:", sig)
else:
    print("Program:", prog)

demos = getattr(prog, "demos", None) or getattr(prog, "_demos", [])
if demos:
    print("\nFew-shot examples used:")
    for i, ex in enumerate(demos[:5], 1):
        q = getattr(ex, "question", str(ex))
        a = getattr(ex, "answer", "")
        print(f"  {i}. Q: {q} -> A: {a}")
else:
    print("\nNo demos attribute found (DSPy may store them differently).")

Program: predict = Predict(StringSignature(question -> reasoning, answer
    instructions='Given the fields `question`, produce the fields `answer`.'
    question = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Question:', 'desc': '${question}'})
    reasoning = Field(annotation=str required=True json_schema_extra={'prefix': "Reasoning: Let's think step by step in order to", 'desc': '${reasoning}', '__dspy_field_type': 'output'})
    answer = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'output', 'prefix': 'Answer:', 'desc': '${answer}'})
))

No demos attribute found (DSPy may store them differently).


## 4. Export optimized prompt as chat completion messages

Extract the optimized prompt in **OpenAI-compatible format** (`messages` with `role` and `content`) so you can use it directly with the RHOAI completion API (or any chat API) without DSPy.

In [17]:
import json

def get_optimized_prompt_as_messages(optimized_module, example_question: str):
    """
    Build chat completion messages (role + content) from the optimized DSPy program
    so you can use them directly with the RHOAI completion API.
    """
    prog = optimized_module.prog
    demos = getattr(prog, "demos", None) or getattr(prog, "_demos", [])
    sig_str = "question -> answer"
    # ChainOfThought adds reasoning; instruction for the model
    system_content = (
        "Answer the following question. "
        "Reason step by step, then give the final answer."
    )
    messages = [{"role": "system", "content": system_content}]

    # Add few-shot examples (user question -> assistant reasoning + answer)
    for ex in demos:
        q = getattr(ex, "question", None)
        a = getattr(ex, "answer", None)
        if q is None or a is None:
            continue
        reasoning = getattr(ex, "reasoning", "")
        assistant_content = f"{reasoning}\n\nAnswer: {a}" if reasoning else str(a)
        messages.append({"role": "user", "content": q})
        messages.append({"role": "assistant", "content": assistant_content})

    # Final user question
    messages.append({"role": "user", "content": example_question})
    return messages


# Export for a concrete question (or use a placeholder for a template)
example_question = "What is the capital of Germany?"
messages = get_optimized_prompt_as_messages(optimized_qa, example_question)

print("Chat completion messages (use with POST .../v1/chat/completions):\n")
print(json.dumps(messages, indent=2))

Chat completion messages (use with POST .../v1/chat/completions):

[
  {
    "role": "system",
    "content": "Answer the following question. Reason step by step, then give the final answer."
  },
  {
    "role": "user",
    "content": "What is the capital of Germany?"
  }
]


In [8]:
# Optional: use DSPy's ChatAdapter if available (exact format used internally)
def get_messages_via_adapter(optimized_module, example_question: str):
    """Use DSPy's ChatAdapter to get the same message format the LM receives."""
    try:
        adapter = dspy.ChatAdapter()
        prog = optimized_module.prog
        sig = getattr(prog, "signature", None) or getattr(prog, "extended_signature", None)
        demos = getattr(prog, "demos", None) or getattr(prog, "_demos", [])
        if sig is not None and hasattr(adapter, "format"):
            # Adapter expects inputs dict; demos as list of Example-like objects
            inputs = {"question": example_question}
            out = adapter.format(signature=sig, demos=demos, inputs=inputs)
            if isinstance(out, list) and out and isinstance(out[0], dict):
                return out
    except Exception as e:
        print("ChatAdapter not available or format differs:", e)
    return None

adapter_messages = get_messages_via_adapter(optimized_qa, example_question)
if adapter_messages:
    print("Messages from DSPy ChatAdapter (internal format):\n")
    print(json.dumps(adapter_messages, indent=2))
else:
    print("Using manual message format above (works with any chat completion API).")

Using manual message format above (works with any chat completion API).


## 5. RAG-driven prompts and optimization

Use a **context + question -> answer** signature so the model answers from supplied context (e.g. retrieved passages). Define a baseline and an optimized RAG program, then compare them on a leaderboard (ranked by eval score).

In [18]:
# RAG signature: model receives context (retrieved passages) + question -> answer
class RAGQA(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought("context, question -> answer")

    def forward(self, context: str, question: str):
        return self.prog(context=context, question=question)

rag_qa = RAGQA()

# Small RAG trainset: (context, question, answer) — context is the "retrieved" text
rag_trainset = [
    dspy.Example(
        context="Paris is the capital and largest city of France. It is known for the Eiffel Tower and the Louvre.",
        question="What is the capital of France?",
        answer="Paris",
    ).with_inputs("context", "question"),
    dspy.Example(
        context="Rome is the capital of Italy. The Colosseum and Vatican City are located there.",
        question="What is the capital of Italy?",
        answer="Rome",
    ).with_inputs("context", "question"),
    dspy.Example(
        context="Berlin is the capital of Germany. It became the capital again after German reunification in 1990.",
        question="What is the capital of Germany?",
        answer="Berlin",
    ).with_inputs("context", "question"),
]

def rag_metric(example, pred, trace=None):
    """Score 1 if the predicted answer matches the gold answer (case-insensitive)."""
    return 1.0 if pred.answer.strip().lower() == example.answer.strip().lower() else 0.0

# Optimize RAG program with BootstrapFewShot
from dspy.teleprompt import BootstrapFewShot
rag_teleprompter = BootstrapFewShot(metric=rag_metric, max_bootstrapped_demos=2, max_labeled_demos=2)
optimized_rag_qa = rag_teleprompter.compile(rag_qa, trainset=rag_trainset)

# Quick test
test_ctx = "Madrid is the capital of Spain. It is home to the Prado Museum."
test_q = "What is the capital of Spain?"
out = optimized_rag_qa(context=test_ctx, question=test_q)
print("RAG test - Question:", test_q, "-> Answer:", out.answer)

 67%|██████▋   | 2/3 [00:04<00:02,  2.07s/it]


Bootstrapped 2 full traces after 2 examples for up to 1 rounds, amounting to 2 attempts.
RAG test - Question: What is the capital of Spain? -> Answer: Madrid


### RAG leaderboard (baseline vs optimized)

Evaluate baseline RAG (ChainOfThought, no few-shot) and optimized RAG (BootstrapFewShot) on the same eval set. Rank by score.

In [19]:
rag_evalset = rag_trainset

def run_rag_eval(program, evalset, metric_fn):
    """Run RAG program on evalset and return mean metric score."""
    scores = []
    for ex in evalset:
        try:
            pred = program(context=ex.context, question=ex.question)
            s = metric_fn(ex, pred)
            scores.append(float(s) if s is not None else 0.0)
        except Exception as e:
            scores.append(0.0)
    return sum(scores) / len(scores) if scores else 0.0

print("Evaluating RAG baseline (ChainOfThought, no few-shot)...")
rag_score_baseline = run_rag_eval(rag_qa, rag_evalset, rag_metric)
print("Evaluating RAG optimized (BootstrapFewShot)...")
rag_score_optimized = run_rag_eval(optimized_rag_qa, rag_evalset, rag_metric)

rag_leaderboard = [
    {"config": "RAG optimized (BootstrapFewShot)", "score": rag_score_optimized, "description": "context+question→answer, few-shot"},
    {"config": "RAG baseline (no optimization)", "score": rag_score_baseline, "description": "context+question→answer, CoT only"},
]
rag_leaderboard.sort(key=lambda x: x["score"], reverse=True)

print("\n" + "=" * 60)
print("RAG LEADERBOARD (ranked by eval score)")
print("=" * 60)
for rank, row in enumerate(rag_leaderboard, 1):
    print(f"  #{rank}  {row['config']:<38}  score = {row['score']:.2%}  ({row['description']})")
print("=" * 60)

Evaluating RAG baseline (ChainOfThought, no few-shot)...
Evaluating RAG optimized (BootstrapFewShot)...

RAG LEADERBOARD (ranked by eval score)
  #1  RAG optimized (BootstrapFewShot)        score = 100.00%  (context+question→answer, few-shot)
  #2  RAG baseline (no optimization)          score = 100.00%  (context+question→answer, CoT only)


In [ ]:
# Optional: RAG leaderboard as pandas DataFrame
try:
    import pandas as pd
    rag_df = pd.DataFrame(rag_leaderboard)[["config", "score", "description"]]
    rag_df.insert(0, "rank", range(1, len(rag_df) + 1))
    rag_df["score"] = rag_df["score"].apply(lambda x: f"{x:.2%}")
    display(rag_df)
except ImportError:
    pass

In [9]:
# Example: call RHOAI completion API directly with the exported messages
def call_rhoai_chat_completion(messages, base_url=None, token=None, model="redhataillama-3-1-8b-instruct"):
    """POST to OpenAI-compatible chat completions using exported messages."""
    import os
    import requests
    base_url = base_url or os.getenv("RHOAI_INFERENCE_URL", "").rstrip("/")
    if "/v1" not in base_url:
        base_url = base_url + "/v1"
    token = token or os.getenv("RHOAI_TOKEN", "")
    url = f"{base_url}/chat/completions"
    resp = requests.post(
        url,
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json={"model": model, "messages": messages, "max_tokens": 256},
        timeout=60,
    )
    resp.raise_for_status()
    data = resp.json()
    return data["choices"][0]["message"]["content"]

# Uncomment to run (requires requests and valid RHOAI endpoint):
# answer = call_rhoai_chat_completion(messages)
# print("Answer from API:", answer)